# Меры риска в автостраховании: сравнение групп по полу и штатам

Курс «Теория выборки и оценка рисков». Данные: 6 773 страховые выплаты (штат, рисковый класс, пол, возраст, сумма выплаты).


In [11]:
# импортируем библиотеки
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import warnings

In [13]:
# отключаем предупреждения FutureWarning:
warnings.simplefilter("ignore", category='PerformanceWarning')

Столбцы:
<li> STATE — код штата
<li> CLASS — рисковый класс (учтен ряд факторов по методике компании)
<li> GENDER — пол (M - мужчины, F - женщины)
<li> AGE — возраст
<li> PAID — размер выплаты (доллары США)

In [2]:
# записываем датасет в переменную:
df = pd.read_excel('data/auto_insurance.xlsx')
# выводим первые 2 строки:
df.head(2)

,STATE,CLASS,GENDER,AGE,PAID
0,STATE 14,C6,M,97,1134.44
1,STATE 15,C6,M,96,3761.24


In [3]:
# делаем заголовки строчными:
df.columns = df.columns.str.lower()
# выводим первые 2 строки:
df.head(2)

,state,class,gender,age,paid
0,STATE 14,C6,M,97,1134.44
1,STATE 15,C6,M,96,3761.24


In [4]:
# выводим основную информацию о датасете:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6773 entries, 0 to 6772
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   state   6773 non-null   object 
 1   class   6773 non-null   object 
 2   gender  6773 non-null   object 
 3   age     6773 non-null   int64  
 4   paid    6773 non-null   float64
dtypes: float64(1), int64(1), object(3)
memory usage: 264.7+ KB


### GENDER

Нам потребуется рассчитать положительную и отрицательные полудисперсии, для которых нет встроенных функций. Поэтому напишем ее сами:

In [9]:
# создаем функцию:
def semivar(data):
    # считаем среднее значение:
    mu = np.mean(data)
    # создаем массивы отдельно положительных и отрицательных отклонений от среднего в каждой строке датасета:
    pos = np.maximum(data - mu, 0)
    neg = np.maximum(mu - data, 0)
    # рассчитываем положительную и отрицательную полудисперсии:
    posvar = np.mean(pos ** 2)
    negvar = np.mean(neg ** 2)
    # возвращаем рассчитанные значения:
    return posvar, negvar

Сначала разделим совокупность по <b>полу</b> и рассчитаем все необходимые по условию меры риска:

In [14]:
# создаем сводную таблицу:
groups = df.pivot_table(index='gender', values='paid', 
                        aggfunc=['var', 
                                 'std', 
                                 lambda x: semivar(x)[0], 
                                 lambda x: semivar(x)[1], 
                                 'max', 
                                 lambda x: np.percentile(x, 95)]).reset_index()
# переименовываем столбцы:
groups.columns = ['Пол', 'Дисперсия', 'СКО', 'Положительная полудисперсия', 
                  'Отрицательная полудисперсия', 'Максимальный ущерб', 'Рисковый капитал']
# выводим сводную таблицу:
groups

,Пол,Дисперсия,СКО,Положительная полудисперсия,Отрицательная полудисперсия,Максимальный ущерб,Рисковый капитал
0,F,7.620810e+06,2760.581523,6.603872e+06,1.013987e+06,60000.00,6448.3935
1,M,6.629054e+06,2574.694894,5.696383e+06,9.310888e+05,59113.78,6303.4650


<b>ВЫВОД:</b><br>
По результатам расчета различных мер риска по всем показателям более рисковой группой являются <b>женщины</b>.

На мой взгляд, для выделенных групп (да и в целом, для всего контекста, т. к. мы говорим об <i>автостраховании</i>) наилучшей мерой риска будет являться <b>рисковый капитал</b>, т. к. он лучше других объясняет наибольшие (с вероятностью 95 % в нашем случае) потери от возможного ущерба, а также не так сильно искажается при наличии <i>выбросов</i>, т. к. является квантилью, а не производным от среднего значения (в отличие от дисперсии, СКО и полудисперсий).

### STATE

Теперь разделим совокупность по <b>географическому положению</b> и снова рассчитаем все необходимые по условию меры риска:

In [17]:
# создаем сводную таблицу:
groups1 = df.pivot_table(index='state', values='paid', 
                         aggfunc=['var', 
                                  'std', 
                                  lambda x: semivar(x)[0], 
                                  lambda x: semivar(x)[1], 
                                  'max', 
                                  lambda x: np.percentile(x, 95)]).reset_index()
# переименовываем столбцы:
groups1.columns = ['Код штата', 'Дисперсия', 'СКО', 'Положительная полудисперсия', 
                   'Отрицательная полудисперсия', 'Максимальный ущерб', 'Рисковый капитал']
# выводим сводную таблицу:
groups1

,Код штата,Дисперсия,СКО,Положительная полудисперсия,Отрицательная полудисперсия,Максимальный ущерб,Рисковый капитал
0,STATE 01,3.941779e+06,1985.391511,3.279568e+06,6.384659e+05,15845.00,4462.1750
1,STATE 02,5.450391e+06,2334.607321,4.584938e+06,8.605951e+05,23231.54,5837.7055
2,STATE 03,8.952699e+06,2992.106079,7.839452e+06,1.087521e+06,23344.38,7470.8455
3,STATE 04,7.332263e+06,2707.815109,6.462816e+06,8.584376e+05,28593.49,6041.3425
4,STATE 06,5.947683e+06,2438.787106,4.771203e+06,1.166917e+06,19348.67,7273.6560
5,STATE 07,1.694645e+07,4116.606786,1.571886e+07,1.164593e+06,60000.00,6414.9680
6,STATE 10,5.324949e+06,2307.585095,4.371115e+06,9.345403e+05,21404.96,6000.6250
7,STATE 11,2.141788e+06,1463.484925,1.243637e+06,6.601751e+05,4635.00,3941.1440
8,STATE 12,7.211720e+06,2685.464616,5.561995e+06,1.620528e+06,16161.53,8471.5250
9,STATE 13,9.847573e+06,3138.084260,8.670503e+06,1.129726e+06,33137.50,6744.7880


Здесь трудно сразу на глаз определить максимальные значения каждого из столбцов, поэтому воспользуемся инструментами питона:

In [19]:
[groups1.loc[groups1[col].idxmax(), 'Код штата'] for col in groups1.columns if col != 'Код штата']

['STATE 07', 'STATE 07', 'STATE 07', 'STATE 12', 'STATE 07', 'STATE 12']

<b>ВЫВОД:</b><br>
По результатам расчета различных мер риска наиболее рисковой группой является <b>STATE 07</b>, т. к. 4 меры риска из 6 у нее наибольшие.

На мой взгляд, для данных выделенных групп наилучшей мерой риска также будет являться <b>рисковый капитал</b>, причем по тем же причинам, уже перечисленным мною в прошлом пункте (повторяться не буду).